# Gold — dim_video_scd1 (SCD 1, poc method)

One row per `video_id`, latest values.
Method (poc `scd_type_1`, helper `_scd`): **changed keys** in the window → re-read their full current rows → metadata `etl_date` / `etl_by` → `add_checksums` → **full**: overwrite / **incremental**: MERGE on `primary_checksum`, update only when `record_checksum` differs → Unknown row → grain check. Widgets: `load_type`, `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). Runs in task `gold_dims_scd1` (nothing depends on it); the live dim is unchanged.

**Changed keys** = videos read in the window (`stg_videos.snapshot_date`) **∪** videos classified in the window (`stg_video_classification.classified_at`): a video classified without a new read still gets its `video_type`. Values from `stg_videos_latest` + `stg_video_classification` + YouTube category names (same rules as `dims.ipynb`).

Not carried over from `dim_video`: `is_available`, `first_seen_at`, `last_seen_at` (they change on every read; `created_at` / `modified_at` are kept but not hashed).

In [ ]:
%run ./_scd

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow (today's reads included)
SILVER, GOLD = dbutils.widgets.get("silver"), dbutils.widgets.get("gold")
LOAD_TYPE = dbutils.widgets.get("load_type")
FROM_DATE, TO_DATE = default_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
FULL = LOAD_TYPE == "full"
print("full: every key" if FULL else f"incremental: keys changed in [{FROM_DATE}, {TO_DATE}) (Vietnam days)")

In [ ]:
CATEGORIES = [("1", "Film & Animation"), ("2", "Autos & Vehicles"), ("10", "Music"), ("15", "Pets & Animals"),
              ("17", "Sports"), ("19", "Travel & Events"), ("20", "Gaming"), ("22", "People & Blogs"),
              ("23", "Comedy"), ("24", "Entertainment"), ("25", "News & Politics"), ("26", "Howto & Style"),
              ("27", "Education"), ("28", "Science & Technology"), ("29", "Nonprofits & Activism")]
categories = spark.createDataFrame(CATEGORIES, "category_id STRING, category_name STRING")
CLASS = f"{SILVER}.stg_video_classification"
cls = (spark.table(CLASS).select("video_id", "video_type", F.col("confidence").alias("video_type_confidence"),
                                 "is_food_review", "classified_at")
       if spark.catalog.tableExists(CLASS)
       else spark.createDataFrame([], "video_id STRING, video_type STRING, video_type_confidence DOUBLE, is_food_review BOOLEAN, classified_at TIMESTAMP"))

keys = None if FULL else changed_keys(
    spark.table(f"{SILVER}.stg_videos").where(in_window(F.col("snapshot_date"), FROM_DATE, TO_DATE)).select("video_id"),
    cls.where(in_window(vn_date("classified_at"), FROM_DATE, TO_DATE)).select("video_id"))
latest = spark.table(f"{SILVER}.stg_videos_latest")
if keys is not None:
    latest = latest.join(keys, "video_id", "left_semi")

def _part(unit):
    x = F.regexp_extract("duration", rf"(\d+){unit}", 1)   # '' when the unit is missing (ANSI: '' can't be cast)
    return F.when(x != "", x.cast("int")).otherwise(F.lit(0))

src = (latest.join(categories, "category_id", "left").join(cls.drop("classified_at"), "video_id", "left")
       .select(F.xxhash64("video_id").alias("video_key"), "video_id", "title", "description", "tags", "category_id",
               "category_name", F.col("duration").alias("duration_iso"),
               F.when(F.col("duration").isNotNull(), 3600 * _part("H") + 60 * _part("M") + _part("S")).alias("duration_seconds"),
               "definition", "has_caption", "thumbnail_url", "video_url", "channel_id", "published_at",
               F.date_format(F.from_utc_timestamp("published_at", TZ), "yyyyMMdd").cast("int").alias("published_date_key"),
               "video_type", "video_type_confidence",
               F.coalesce("is_food_review", F.lit(True)).alias("is_food_review"),
               "created_at", F.col("snapshot_ts").alias("modified_at"))
       .withColumn("etl_date", F.current_timestamp())
       .withColumn("etl_by", F.lit("Databricks")))
src = add_checksums(src, ["video_id"], exclude=["video_key", "created_at", "modified_at"])

In [ ]:
load_scd1(src, f"{GOLD}.dim_video_scd1", """
        video_key BIGINT, video_id STRING, title STRING, description STRING, tags ARRAY<STRING>, category_id STRING,
        category_name STRING, duration_iso STRING, duration_seconds INT, definition STRING, has_caption BOOLEAN,
        thumbnail_url STRING, video_url STRING, channel_id STRING, published_at TIMESTAMP, published_date_key INT,
        video_type STRING, video_type_confidence DOUBLE, is_food_review BOOLEAN,
        created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING""",
    "Video dimension, SCD 1 (poc checksum method). video_key = xxhash64(video_id), -1 = Unknown.",
    ["video_id"], LOAD_TYPE, {"video_key": "-1", "title": "'Unknown'", "category_name": "'Unknown'",
                              "etl_date": "current_timestamp()", "etl_by": "'Databricks'"})